# Evaluation

This notebook is self-contained and should be run on https://jupyter-hub.io-ancotel.local:8443 for open-source models (GPU), or anywhere else for API-only models.

In [ ]:
from __future__ import annotations

import asyncio
import datetime
import gc
import math
import os
import re
import subprocess
import unicodedata
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

In [ ]:
# Current time for output file naming
cet = datetime.timezone(datetime.timedelta(hours=1))
now = datetime.datetime.now(tz=cet).strftime("%Y%m%d_%H%M%S")

In [ ]:
# GPU configuration
GPU_MODE = "single"  # "single" (best free GPU), "multi" (N best), "all" (every GPU)
NUM_GPUS = None  # only used when GPU_MODE="multi"

try:
    _smi = subprocess.run(
        [
            "/usr/bin/nvidia-smi",
            "--query-gpu=memory.free",
            "--format=csv,nounits,noheader",
        ],
        capture_output=True,
        text=True,
        check=True,
    )
    _free = [int(x) for x in _smi.stdout.strip().splitlines()]

    if GPU_MODE == "single":
        _gpu = _free.index(max(_free))
        os.environ["CUDA_VISIBLE_DEVICES"] = str(_gpu)
        print(f"Using physical GPU {_gpu} ({_free[_gpu]} MiB free)")
    elif GPU_MODE == "multi":
        _ranked = sorted(range(len(_free)), key=lambda i: _free[i], reverse=True)
        _selected = _ranked[:NUM_GPUS]
        os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in _selected)
        print(
            f"Using physical GPUs {_selected} (free: {[_free[g] for g in _selected]} MiB)"
        )
    elif GPU_MODE == "all":
        print(f"Using all {len(_free)} GPUs (free: {_free} MiB)")
    else:
        msg = f"Unknown GPU_MODE: {GPU_MODE!r}"
        raise ValueError(msg)
except FileNotFoundError:
    print("No NVIDIA GPU detected — API-only mode")

In [ ]:
# tiktoken configuration
repo_root = Path.cwd().parent  # adjust as needed
os.environ["TIKTOKEN_CACHE_DIR"] = str(repo_root / "tiktoken_cache")

In [ ]:
import numpy as np
import openai
import pandas as pd
import seaborn as sns
import tiktoken
import torch
import transformers
from datasets import load_dataset
from dotenv import load_dotenv
from matplotlib import pyplot as plt
from openai import AsyncOpenAI
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [ ]:
# Filter which models to evaluate: "all", "api", "hf", or "rq3"
# RQ3 mode reruns a fixed LLM on the EASE@250 and SASRec@250 prompt files.
RUN_MODE = "rq3"
RQ3_RERANKER_MODEL = "claude-opus-4-6"

CONFIG = {
    "paths": {
        "data": Path("../data/processed"),
        "output": Path("../data/output"),
        "ft_adapter": Path("../models/Qwen2.5-7B-Instruct-FT"),
    },
    "model": {
        "attn_implementation": "flash_attention_2",
    },
    "evaluation": {
        "batch_size": 32,
        "batch_size_by_ccount": {  # HF models only (GPU memory constraint)
            "c0": 32,
            "c250": 16,
            "c500": 8,
            "c1000": 4,
            "cALL": 2,
        },
        "max_input_length": 8192,
        "max_new_tokens": 256,
        "k_values": [1, 5, 10],
        "rq3_prompt_sources": {
            "ease_c250": "test_prompt_examples_cf_EASE_c250_r10.jsonl",
            "sasrec_c250": "test_prompt_examples_seq_SASRec_c250_r10.jsonl",
        },
        "temperature": 0.0,
        "top_p": 1.0,
    },
    "api": {
        "max_concurrent": 2,
        "encoding": "o200k_base",
    },
    "seed": 42,
}

In [ ]:
@dataclass
class ModelConfig:
    """Configuration for a single model to evaluate."""

    name: str  # display name for output
    model_type: str  # "local_hf" | "local_hf_ft" | "api"
    model_id: str  # HF repo ID or API model name
    candidate_counts: list[str] = field(default_factory=list)
    context_window: int = 128_000
    quantization: str | None = None  # "4bit_bnb" or None
    adapter_path: str | None = None  # LoRA adapter path (FT only)
    dtype: str = "bfloat16"
    supports_system_role: bool = True  # False for Gemma etc.
    supports_sampling: bool = (
        True  # False for reasoning models that reject temperature/top_p
    )
    extra_api_params: dict = field(default_factory=dict)  # extra params for extra_body
    temperature_override: float | None = None  # override default temperature
    max_new_tokens_override: int | None = None  # override default max_new_tokens


MODEL_REGISTRY: list[ModelConfig] = [
    # Open-source (GPU server)
    ModelConfig(
        name="Llama-3.2-3B",
        model_type="local_hf",
        model_id="meta-llama/Llama-3.2-3B-Instruct",
        candidate_counts=["c0", "c250", "c500"],
    ),
    ModelConfig(
        name="Llama-3.1-8B",
        model_type="local_hf",
        model_id="meta-llama/Llama-3.1-8B-Instruct",
        candidate_counts=["c0", "c250", "c500"],
    ),
    ModelConfig(
        name="Llama-3.3-70B",
        model_type="local_hf",
        model_id="meta-llama/Llama-3.3-70B-Instruct",
        candidate_counts=["c0", "c250", "c500"],
        quantization="4bit_bnb",
    ),
    ModelConfig(
        name="Qwen2.5-7B",
        model_type="local_hf",
        model_id="Qwen/Qwen2.5-7B-Instruct",
        candidate_counts=["c0", "c250", "c500"],
    ),
    ModelConfig(
        name="Qwen2.5-7B-FT",
        model_type="local_hf_ft",
        model_id="Qwen/Qwen2.5-7B-Instruct",
        candidate_counts=["c250"],
        adapter_path=CONFIG["paths"]["ft_adapter"],
    ),
    ModelConfig(
        name="Gemma-2-9B",
        model_type="local_hf",
        model_id="google/gemma-2-9b-it",
        candidate_counts=["c0", "c250", "c500"],
        supports_system_role=False,
    ),
    # API models (via LiteLLM)
    ModelConfig(
        name="infobip-gpt-4-1",
        model_type="api",
        model_id="infobip-gpt-4-1",
        candidate_counts=["c0", "c250", "c500", "c1000", "cALL"],
    ),
    ModelConfig(
        name="infobip-gpt-4-1-mini",
        model_type="api",
        model_id="infobip-gpt-4-1-mini",
        candidate_counts=["c0", "c250", "c500", "c1000", "cALL"],
    ),
    ModelConfig(
        name="gpt-5.2",
        model_type="api",
        model_id="gpt-5.2",
        candidate_counts=["c0", "c250", "c500", "c1000", "cALL"],
        extra_api_params={"reasoning_effort": "none"},
    ),
    ModelConfig(
        name="gpt-5.2",
        model_type="api",
        model_id="gpt-5.2",
        candidate_counts=["c0"],
        extra_api_params={"reasoning_effort": "none"},
    ),
    ModelConfig(
        name="claude-opus-4-6",
        model_type="api",
        model_id="claude-opus-4-6",
        candidate_counts=["c0", "c250", "c500", "c1000", "cALL"],
        context_window=200_000,
    ),
]

In [ ]:
# Seed
SEED = CONFIG["seed"]
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)

# Enable TF32 for faster matmul on Ampere+ GPUs
if torch.cuda.is_available():
    torch.backends.cuda.matmul.fp32_precision = "tf32"
    torch.backends.cudnn.conv.fp32_precision = "tf32"

# Environment
load_dotenv()

# API client
base_url = os.getenv("LITELLM_ENDPOINT", "").rstrip("/")
if base_url and not base_url.endswith("/v1"):
    base_url += "/v1"
api_client = AsyncOpenAI(
    base_url=base_url,
    api_key=os.getenv("LITELLM_API_KEY"),
)
print(f"API client base_url: {base_url}")

# Tiktoken encoder for entropy fallback
enc = tiktoken.get_encoding(CONFIG["api"]["encoding"])

## Utility functions

> **Note:** Functions below are copied from `src/stability/` to keep this notebook
> self-contained for remote JupyterHub execution.

In [ ]:
# Copied from src/stability/utils.py
def canonicalize(s: str | None) -> str | None:
    """Return a canonicalized version of the string for comparison."""
    if s is None:
        return None
    if not isinstance(s, str):
        s = str(s)
    s = s.replace("\u2014", "-").replace("\u2013", "-")
    s = "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )
    result = s.strip().strip('"').strip("'").rstrip(".,:;!?").lower()
    result = re.sub(r"\s+", " ", result)
    return result.replace("&", "and")

In [ ]:
# Copied from src/stability/generation.py
def extract_pred_items(response: str, max_items: int = 10) -> list[str]:
    """Extract predicted items (movie titles) from raw LLM response text."""
    response = response.strip()
    lines = [
        re.sub(r"^\d+[.)]\s*", "", ln).strip(" -*•\t")
        for ln in response.splitlines()
        if ln.strip()
    ]

    items: list[str] = []
    for ln in lines:
        if ln and 2 <= len(ln) <= 120:
            items.append(ln)
        if len(items) >= max_items:
            break

    seen: set[str | None] = set()
    dedup: list[str] = []
    for it in items:
        it_canon = canonicalize(it)
        if it_canon and it_canon not in seen:
            seen.add(it_canon)
            dedup.append(it)
    return dedup[:max_items]

In [ ]:
# Copied from src/stability/generation.py
def detect_assistant_separator(
    tokenizer: Any,  # noqa: ANN401
    supports_system_role: bool = True,
) -> str:
    """Return the separator string for the assistant role from the chat template."""
    test_messages = [
        {"role": "user", "content": "test"},
        {"role": "assistant", "content": "test"},
    ]
    if supports_system_role:
        test_messages.insert(0, {"role": "system", "content": "test"})
    formatted = tokenizer.apply_chat_template(test_messages, tokenize=False)
    patterns = [
        ("llama3", "<|start_header_id|>assistant<|end_header_id|>\n\n"),
        ("qwen", "<|im_start|>assistant\n"),
        ("phi3", "<|assistant|>\n"),
        ("gemma", "model\n"),
        ("mistral", "[/INST]"),
    ]
    for name, pattern in patterns:
        if pattern in formatted:
            print(f"Detected {name} chat template, separator: {pattern!r}")
            return pattern

    # Fallback, look for the assistant marker
    if "<|start_header_id|>assistant<|end_header_id|>" in formatted:
        return "<|start_header_id|>assistant<|end_header_id|>\n\n"
    if "<|im_start|>assistant" in formatted:
        return "<|im_start|>assistant\n"
    if "<|assistant|>" in formatted:
        return "<|assistant|>\n"
    if "[/INST]" in formatted:
        return "[/INST]"

    # If nothing found, raise with formatted text for manual inspection
    msg = (
        "Could not automatically detect the assistant separator.\n"
        f"Formatted test messages:\n{formatted}\n"
        "Please inspect the above output to determine the correct separator."
    )
    raise ValueError(msg)

In [ ]:
# Copied from src/stability/metrics.py (standalone version)
def recommendation_metrics(
    predictions: list[str],
    ground_truth: list[str],
    k_values: list[int],
) -> dict[str, float]:
    """Return hit_rate@K, mrr@K, precision@K, recall@K, f1@K, ndcg@K."""
    metrics: dict[str, float] = {}
    if not ground_truth:
        for k in k_values:
            for name in ["hit_rate", "mrr", "precision", "recall", "f1", "ndcg"]:
                metrics[f"{name}@{k}"] = np.nan
        return metrics

    truth_canon = [canonicalize(t) for t in ground_truth if t]
    pred_canon = [canonicalize(p) for p in predictions if p]

    for k in k_values:
        topk = pred_canon[:k]
        relevance = np.array([1 if p in truth_canon else 0 for p in topk])

        hit_rate_k = 1.0 if relevance.sum() > 0 else 0.0
        mrr_k = 0.0
        if relevance.any():
            mrr_k = 1.0 / (np.argmax(relevance) + 1)

        tp = relevance.sum()
        precision_k = tp / k
        recall_k = tp / len(truth_canon)
        f1_k = (
            2 * precision_k * recall_k / (precision_k + recall_k)
            if (precision_k + recall_k) > 0
            else 0.0
        )

        # NDCG (textbook: ideal ranking places all ground-truth items at top)
        dcg = np.sum(relevance / np.log2(np.arange(2, len(relevance) + 2)))
        n_relevant = min(len(truth_canon), k)
        ideal_relevance = np.zeros(k)
        ideal_relevance[:n_relevant] = 1
        idcg = np.sum(ideal_relevance / np.log2(np.arange(2, k + 2)))
        ndcg_k = dcg / idcg if idcg > 0 else 0.0

        metrics[f"hit_rate@{k}"] = hit_rate_k
        metrics[f"mrr@{k}"] = mrr_k
        metrics[f"precision@{k}"] = precision_k
        metrics[f"recall@{k}"] = recall_k
        metrics[f"f1@{k}"] = f1_k
        metrics[f"ndcg@{k}"] = ndcg_k
    return metrics

In [ ]:
def compute_text_entropy(text: str, encoder: tiktoken.Encoding) -> dict[str, float]:
    """Compute token-level entropy metrics from text using a tiktoken encoder."""
    token_ids = encoder.encode(text)
    if not token_ids:
        return {"entropy": 0.0, "normalized_entropy": 0.0, "unique_token_ratio": 0.0}
    unique, counts = np.unique(token_ids, return_counts=True)
    probs = counts / counts.sum()
    entropy = float(-np.sum(probs * np.log2(probs)))
    max_entropy = math.log2(len(unique)) if len(unique) > 1 else 1.0
    return {
        "entropy": entropy,
        "normalized_entropy": entropy / max_entropy if max_entropy > 0 else 0.0,
        "unique_token_ratio": len(unique) / len(token_ids),
    }

In [ ]:
# Copied from src/stability/utils.py
def bootstrap_mean_ci(
    values: np.ndarray,
    rng: np.random.Generator,
    n_boot: int = 1000,
    ci: float = 95.0,
) -> tuple[float, float, float]:
    """Return mean and bootstrap confidence interval for the given values."""
    vals = values[np.isfinite(values)]
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    mean_val = float(np.mean(vals))
    if len(vals) == 1:
        return mean_val, mean_val, mean_val

    boots = [
        rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)
    ]
    low = np.percentile(boots, (100 - ci) / 2)
    high = np.percentile(boots, 100 - (100 - ci) / 2)
    return mean_val, float(low), float(high)

## Prompt builders

In [ ]:
def merge_system_into_user(messages: list[dict[str, str]]) -> list[dict[str, str]]:
    """Merge system message content into the first user message."""
    system_parts = [m["content"] for m in messages if m["role"] == "system"]
    other = [m for m in messages if m["role"] != "system"]
    if not system_parts:
        return other
    prefix = "\n\n".join(system_parts)
    return [
        {**msg, "content": prefix + "\n\n" + msg["content"]}
        if msg["role"] == "user"
        else msg
        for msg in other
    ]


def build_hf_prompt(
    messages: list[dict[str, str]],
    tokenizer: Any,  # noqa: ANN401
    sep: str,
    supports_system_role: bool = True,
) -> str:
    """Apply chat template and strip at the assistant separator."""
    if not supports_system_role:
        messages = merge_system_into_user(messages)
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    # Keep everything up to and including the separator
    if sep in formatted:
        idx = formatted.index(sep) + len(sep)
        return formatted[:idx]
    return formatted

In [ ]:
def build_api_messages(messages: list[dict[str, str]]) -> list[dict[str, str]]:
    """Strip the assistant message, return system+user for chat completions."""
    return [m for m in messages if m["role"] != "assistant"]

## Generators

In [ ]:
@torch.no_grad()
def generate_hf(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompts: list[str],
    max_new_tokens: int = 256,
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_input_length: int = CONFIG["evaluation"]["max_input_length"],
) -> list[dict[str, str | float]]:
    """Batch HF generation with entropy computation."""
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_input_length,
    ).to(model.device)

    if inputs["input_ids"].shape[1] >= max_input_length:
        print(f"  WARNING: Input truncated to {max_input_length} tokens")

    gen_kwargs = {
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
        "max_new_tokens": max_new_tokens,
        "output_scores": True,
        "return_dict_in_generate": True,
    }
    if temperature > 0:
        gen_kwargs.update(
            {"do_sample": True, "temperature": temperature, "top_p": top_p}
        )
    else:
        gen_kwargs["do_sample"] = False

    outputs = model.generate(**inputs, **gen_kwargs)
    input_len = inputs["input_ids"].shape[1]

    results: list[dict[str, str | float]] = []
    for i, output_ids in enumerate(outputs.sequences):
        response_ids = output_ids[input_len:]
        response = tokenizer.decode(response_ids, skip_special_tokens=True).strip()

        # Compute entropy from logits
        logits = torch.stack(outputs.scores, dim=1)[i, :, :]
        probs = torch.nn.functional.softmax(logits, dim=-1)
        token_entropy = -torch.sum(probs * torch.clamp(probs, min=1e-10).log2(), dim=-1)
        avg_entropy = token_entropy.mean().item()
        norm_entropy = avg_entropy / math.log2(probs.shape[-1])

        results.append(
            {
                "response": response,
                "entropy": avg_entropy,
                "normalized_entropy": norm_entropy,
                "unique_token_ratio": len(set(response_ids.tolist()))
                / max(len(response_ids), 1),
            }
        )
    return results

In [ ]:
async def generate_api(
    client: AsyncOpenAI,
    model: str,
    messages_batch: list[list[dict[str, str]]],
    max_new_tokens: int = 256,
    temperature: float = 0.0,
    top_p: float = 1.0,
    max_concurrent: int = 10,
    extra_body: dict | None = None,
    progress_prefix: str | None = None,
) -> list[dict[str, str | float]]:
    """Async chat completions with entropy fallback."""
    semaphore = asyncio.Semaphore(max_concurrent)
    total = len(messages_batch)
    completed = 0

    async def call_api(msgs: list[dict[str, str]], idx: int) -> tuple[int, dict]:
        nonlocal completed
        async with semaphore:
            text = ""
            entropy_metrics = {
                "entropy": 0.0,
                "normalized_entropy": 0.0,
                "unique_token_ratio": 0.0,
            }
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    create_kwargs: dict[str, Any] = {
                        "model": model,
                        "messages": msgs,
                        "temperature": temperature,
                        "max_tokens": max_new_tokens,
                        "seed": SEED,
                        **(extra_body or {}),
                    }
                    # Only send top_p when it differs from the default (1.0);
                    # Claude/Bedrock rejects having both temperature and top_p set.
                    if top_p != 1.0:
                        create_kwargs["top_p"] = top_p
                    response = await client.chat.completions.create(**create_kwargs)
                    text = (response.choices[0].message.content or "").strip()
                    entropy_metrics = compute_text_entropy(text, enc)
                    break
                except (
                    openai.APIError,
                    openai.APIConnectionError,
                    openai.RateLimitError,
                    openai.APITimeoutError,
                ) as e:
                    wait = 2**attempt
                    if attempt < max_retries - 1:
                        print(
                            f"API error for idx {idx} (attempt {attempt + 1}): {e}, retrying in {wait}s"
                        )
                        await asyncio.sleep(wait)
                    else:
                        print(
                            f"API error for idx {idx} (attempt {attempt + 1}): {e}, giving up"
                        )
            completed += 1
            if progress_prefix:
                print(
                    f"{progress_prefix} | {completed}/{total} ({completed * 100 // total}%)",
                    end="\r",
                )
            return idx, {"response": text, **entropy_metrics}

    tasks = [call_api(msgs, i) for i, msgs in enumerate(messages_batch)]
    results = await asyncio.gather(*tasks)
    if progress_prefix:
        print()  # newline after final \r
    results_sorted = sorted(results, key=lambda x: x[0])
    return [r for _, r in results_sorted]

## Model loading / unloading

In [ ]:
def load_hf_model(cfg: ModelConfig) -> tuple[AutoModelForCausalLM, Any]:
    """Load a HuggingFace model (optionally with 4-bit quant and/or LoRA)."""
    dtype = getattr(torch, cfg.dtype)
    load_kwargs = {
        "dtype": dtype,
        "device_map": "auto",
        "trust_remote_code": True,
        "attn_implementation": CONFIG["model"]["attn_implementation"],
    }

    if cfg.quantization == "4bit_bnb":
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=dtype,
        )

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_id, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id
    tokenizer.padding_side = "left"  # required for decoder-only batch generation

    model = AutoModelForCausalLM.from_pretrained(cfg.model_id, **load_kwargs)

    # Merge LoRA adapter for fine-tuned models
    if cfg.model_type == "local_hf_ft" and cfg.adapter_path:
        model = PeftModel.from_pretrained(model, cfg.adapter_path)
        model = model.merge_and_unload()
        print(f"  LoRA adapter merged from {cfg.adapter_path}")

    model.eval()

    # Sync pad/eos tokens
    for c in (model.config, model.generation_config):
        c.pad_token_id = tokenizer.pad_token_id
        c.eos_token_id = tokenizer.eos_token_id

    # Reset sampling defaults to suppress "generation flags are not valid"
    # warnings when do_sample=False (some models ship with temperature!=1.0)
    model.generation_config.temperature = 1.0
    model.generation_config.top_p = 1.0

    print(f"  Loaded {cfg.name} ({cfg.model_id}, quant={cfg.quantization})")
    return model, tokenizer


def unload_model(model: Any, tokenizer: Any) -> None:  # noqa: ANN401
    """Free GPU memory."""
    del model
    del tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("  Model unloaded, CUDA cache cleared")

## Batch processing

In [ ]:
def process_results(
    model_name: str,
    model_type: str,
    model_id: str,
    n_candidates: str,
    generation_results: list[dict[str, str | float]],
    ground_truth_batch: list[list[str]],
    prompt_indices: list[int],
    k_values: list[int],
    prompt_source: str | None = None,
    retriever: str | None = None,
) -> list[dict]:
    """Combine generation output with recommendation metrics."""
    records = []
    for i, res in enumerate(generation_results):
        response = res["response"]
        pred_items = extract_pred_items(response, max_items=10)
        gt = ground_truth_batch[i]
        metrics = recommendation_metrics(pred_items, gt, k_values)
        records.append(
            {
                "model": model_name,
                "model_type": model_type,
                "model_id": model_id,
                "n_candidates": n_candidates,
                "prompt_source": prompt_source or n_candidates,
                "retriever": retriever,
                "prompt_idx": prompt_indices[i],
                "response": response,
                "pred_items": pred_items,
                "num_pred_items": len(pred_items),
                "ground_truth": gt,
                "num_ground_truth": len(gt),
                "entropy": res.get("entropy", 0.0),
                "normalized_entropy": res.get("normalized_entropy", 0.0),
                "unique_token_ratio": res.get("unique_token_ratio", 0.0),
                **metrics,
            }
        )
    return records

## Main evaluation loop

In [ ]:
[model.name for model in MODEL_REGISTRY]

In [ ]:
# Quick sanity check: run a single prompt for any model
# Set these to test a specific model + candidate count:
TEST_MODEL_NAME = "gpt-5.2"  # any name from MODEL_REGISTRY
TEST_CCOUNT = "cALL"  # any candidate count that model supports
TEST_EXAMPLE_IDX = 0  # index into the dataset

test_cfg = next(m for m in MODEL_REGISTRY if m.name == TEST_MODEL_NAME)
_test_fname = CONFIG["paths"]["data"] / f"test_prompt_examples_{TEST_CCOUNT}_r10.jsonl"
_test_ds = load_dataset("json", data_files=_test_fname.as_posix())["train"]
test_example = _test_ds[TEST_EXAMPLE_IDX]

if test_cfg.model_type == "api":
    test_messages = build_api_messages(test_example["messages"])
    print("=== API Messages ===")
    for msg in test_messages:
        print(f"\n[{msg['role']}]:\n{msg['content']}...")
    test_results = await generate_api(
        client=api_client,
        model=test_cfg.model_id,
        messages_batch=[test_messages],
        max_new_tokens=CONFIG["evaluation"]["max_new_tokens"],
        temperature=CONFIG["evaluation"]["temperature"],
        top_p=CONFIG["evaluation"]["top_p"],
        max_concurrent=1,
        extra_body=test_cfg.extra_api_params or None,
    )
else:
    test_model, test_tokenizer = load_hf_model(test_cfg)
    test_sep = detect_assistant_separator(test_tokenizer, test_cfg.supports_system_role)
    test_prompt = build_hf_prompt(
        test_example["messages"],
        test_tokenizer,
        test_sep,
        test_cfg.supports_system_role,
    )
    print(f"=== HF Prompt ({len(test_prompt)} chars) ===")
    print(test_prompt + "\n...\n" + test_prompt)
    test_results = generate_hf(
        model=test_model,
        tokenizer=test_tokenizer,
        prompts=[test_prompt],
        max_new_tokens=CONFIG["evaluation"]["max_new_tokens"],
        temperature=CONFIG["evaluation"]["temperature"],
        top_p=CONFIG["evaluation"]["top_p"],
    )
    unload_model(test_model, test_tokenizer)

print("\n=== Response ===")
print(test_results[0]["response"])
print("\n=== Extracted items ===")
print(extract_pred_items(test_results[0]["response"]))
print("\n=== Ground truth ===")
print(test_example["ground_truth"])
print("\n=== Metrics ===")
test_metrics = recommendation_metrics(
    extract_pred_items(test_results[0]["response"]),
    test_example["ground_truth"],
    CONFIG["evaluation"]["k_values"],
)
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
DATA_PATH = CONFIG["paths"]["data"]
OUTPUT_PATH = CONFIG["paths"]["output"]
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

LOG_FILE = OUTPUT_PATH / f"evaluation_{now}.log"


def log(msg: str, end: str = "\n") -> None:
    """Print to stdout and append to log file."""
    print(msg, end=end)
    if end == "\n":
        with LOG_FILE.open("a") as f:
            f.write(msg + "\n")


EMPTY_RESULT = {
    "response": "",
    "entropy": 0.0,
    "normalized_entropy": 0.0,
    "unique_token_ratio": 0.0,
}


def run_hf_batch(
    prompts: list[str],
    model: Any,  # noqa: ANN401
    tokenizer: Any,  # noqa: ANN401
    max_new_tokens: int,
    temperature: float,
    top_p: float,
) -> list[dict[str, str | float]] | None:
    """Run HF generation. Returns list of results, or None on failure."""
    try:
        return generate_hf(
            model=model,
            tokenizer=tokenizer,
            prompts=prompts,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
        )
    except (RuntimeError, OSError):
        torch.cuda.empty_cache()
        return None


async def run_api_batch(
    messages_batch: list[list[dict[str, str]]],
    cfg: ModelConfig,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
    progress_prefix: str | None = None,
) -> list[dict[str, str | float]] | None:
    """Run API generation. Returns list of results, or None on failure."""
    try:
        return await generate_api(
            client=api_client,
            model=cfg.model_id,
            messages_batch=messages_batch,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            max_concurrent=CONFIG["api"]["max_concurrent"],
            extra_body=cfg.extra_api_params or None,
            progress_prefix=progress_prefix,
        )
    except (RuntimeError, OSError):
        return None


def retriever_from_prompt_source(prompt_source: str) -> str:
    """Map a prompt-source key to its first-stage retriever label."""
    if prompt_source == "ease_c250":
        return "EASE"
    if prompt_source == "sasrec_c250":
        return "SASRec"
    return "Semantic"


all_records: list[dict] = []
checkpoint_path = OUTPUT_PATH / f"evaluation_results_partial_{now}.parquet"
completed_pairs: set[tuple[str, str]] = set()

# RQ3 mode deliberately starts a fresh checkpoint unless this exact timestamp's
# checkpoint exists, because it has different pair keys from the main matrix.
if checkpoint_path.exists():
    df_ckpt = pd.read_parquet(checkpoint_path)
    all_records = df_ckpt.to_dict("records")
    completed_pairs = set(
        zip(
            df_ckpt["model"],
            df_ckpt.get("prompt_source", df_ckpt["n_candidates"]),
            strict=True,
        )
    )
    log(
        f"Resumed from checkpoint: {len(all_records)} records, "
        f"{len(completed_pairs)} (model, prompt_source) pairs done"
    )
elif RUN_MODE != "rq3" and (
    existing_partials := sorted(
        OUTPUT_PATH.glob("evaluation_results_partial_*.parquet")
    )
):
    checkpoint_path = existing_partials[-1]
    df_ckpt = pd.read_parquet(checkpoint_path)
    all_records = df_ckpt.to_dict("records")
    completed_pairs = set(zip(df_ckpt["model"], df_ckpt["n_candidates"], strict=True))
    log(
        f"Resumed from latest partial {checkpoint_path.name}: "
        f"{len(all_records)} records, {len(completed_pairs)} (model, ccount) pairs done"
    )

models_to_run = MODEL_REGISTRY
if RUN_MODE == "api":
    models_to_run = [m for m in MODEL_REGISTRY if m.model_type == "api"]
elif RUN_MODE == "hf":
    models_to_run = [m for m in MODEL_REGISTRY if m.model_type != "api"]
elif RUN_MODE == "rq3":
    matches = [m for m in MODEL_REGISTRY if m.name == RQ3_RERANKER_MODEL]
    if not matches:
        msg = f"RQ3_RERANKER_MODEL not in registry: {RQ3_RERANKER_MODEL}"
        raise ValueError(msg)
    models_to_run = [matches[0]]
    log(f"RQ3 mode: fixed reranker = {RQ3_RERANKER_MODEL}")

active_model_names = {m.name for m in models_to_run}
if all_records:
    before_filter = len(all_records)
    all_records = [r for r in all_records if r.get("model") in active_model_names]
    if len(all_records) != before_filter:
        log(
            f"Dropped {before_filter - len(all_records)} checkpoint rows for models "
            f"not in active registry: {sorted(active_model_names)}"
        )
    if all_records:
        df_ckpt = pd.DataFrame.from_records(all_records)
        key_col = (
            "prompt_source"
            if RUN_MODE == "rq3" and "prompt_source" in df_ckpt
            else "n_candidates"
        )
        completed_pairs = set(zip(df_ckpt["model"], df_ckpt[key_col], strict=True))
    else:
        completed_pairs = set()

# Load all datasets once. Main mode is keyed by candidate count; RQ3 is keyed by
# prompt-source because every source is c250 but uses a different retriever.
datasets: dict[str, list[dict]] = {}
if RUN_MODE == "rq3":
    for prompt_source, filename in CONFIG["evaluation"]["rq3_prompt_sources"].items():
        fname = DATA_PATH / filename
        ds = load_dataset("json", data_files=fname.as_posix())["train"]
        datasets[prompt_source] = ds
        log(f"Loaded {prompt_source}: {len(ds)} examples from {fname.name}")
else:
    all_ccounts = sorted({cc for cfg in models_to_run for cc in cfg.candidate_counts})
    for cc in all_ccounts:
        fname = DATA_PATH / f"test_prompt_examples_{cc}_r10.jsonl"
        ds = load_dataset("json", data_files=fname.as_posix())["train"]
        datasets[cc] = ds
        log(f"Loaded {cc}: {len(ds)} examples from {fname.name}")

batch_size_map = CONFIG["evaluation"]["batch_size_by_ccount"]
default_batch_size = CONFIG["evaluation"]["batch_size"]
k_values = CONFIG["evaluation"]["k_values"]
max_new_tokens = CONFIG["evaluation"]["max_new_tokens"]
temperature = CONFIG["evaluation"]["temperature"]
top_p = CONFIG["evaluation"]["top_p"]

for cfg in models_to_run:
    prompt_sources = list(datasets) if RUN_MODE == "rq3" else cfg.candidate_counts
    log(f"\n{'=' * 60}")
    log(f"Model: {cfg.name} ({cfg.model_type})")
    log(f"Prompt sources: {prompt_sources}")
    log(f"{'=' * 60}")

    model = tokenizer = sep = None
    if cfg.model_type in ("local_hf", "local_hf_ft"):
        model, tokenizer = load_hf_model(cfg)
        sep = detect_assistant_separator(tokenizer, cfg.supports_system_role)

    for prompt_source in prompt_sources:
        pair_key = (cfg.name, prompt_source)
        if pair_key in completed_pairs:
            log(f"  [{cfg.name}] {prompt_source}: already completed, skipping")
            continue

        ds = datasets[prompt_source]
        n_examples = len(ds)
        n_candidates = "c250" if RUN_MODE == "rq3" else prompt_source
        retriever = retriever_from_prompt_source(prompt_source)

        if cfg.model_type == "api":
            batch_size = n_examples
        else:
            batch_size = batch_size_map.get(n_candidates, default_batch_size)

        n_batches = math.ceil(n_examples / batch_size)
        log(
            f"  [{cfg.name}] {prompt_source} | {n_examples} examples, batch_size={batch_size}"
        )

        for batch_idx, batch_start in enumerate(range(0, n_examples, batch_size)):
            batch_end = min(batch_start + batch_size, n_examples)
            batch = ds[batch_start:batch_end]
            prompt_indices = list(range(batch_start, batch_end))
            cur_batch_size = batch_end - batch_start

            is_local = cfg.model_type in ("local_hf", "local_hf_ft")
            if is_local:
                prompts = [
                    build_hf_prompt(msgs, tokenizer, sep, cfg.supports_system_role)
                    for msgs in batch["messages"]
                ]
                pct = (batch_idx + 1) * 100 // n_batches
                print(
                    f"  [{cfg.name}] {prompt_source} | batch {batch_idx + 1}/{n_batches} ({pct}%)",
                    end="\r",
                )
            else:
                messages_batch = [
                    build_api_messages(msgs) for msgs in batch["messages"]
                ]

            cur_temperature = (
                cfg.temperature_override
                if cfg.temperature_override is not None
                else temperature
            )
            cur_max_new_tokens = (
                cfg.max_new_tokens_override
                if cfg.max_new_tokens_override is not None
                else max_new_tokens
            )

            if is_local:
                gen_results = run_hf_batch(
                    prompts,
                    model,
                    tokenizer,
                    cur_max_new_tokens,
                    cur_temperature,
                    top_p,
                )
            else:
                gen_results = await run_api_batch(
                    messages_batch,
                    cfg,
                    cur_max_new_tokens,
                    cur_temperature,
                    top_p,
                    progress_prefix=f"  [{cfg.name}] {prompt_source}",
                )

            if gen_results is None:
                log(
                    f"  [{cfg.name}] {prompt_source} | batch {batch_idx + 1}: "
                    f"failed, retrying {cur_batch_size} examples one-by-one"
                )
                gen_results = []
                for i in range(cur_batch_size):
                    if is_local:
                        res = run_hf_batch(
                            [prompts[i]],
                            model,
                            tokenizer,
                            cur_max_new_tokens,
                            cur_temperature,
                            top_p,
                        )
                    else:
                        res = await run_api_batch(
                            [messages_batch[i]],
                            cfg,
                            cur_max_new_tokens,
                            cur_temperature,
                            top_p,
                        )
                    if res is not None:
                        gen_results.append(res[0])
                    else:
                        log(
                            f"  [{cfg.name}] {prompt_source} | example {prompt_indices[i]}: retry failed, using empty response"
                        )
                        gen_results.append(EMPTY_RESULT)

            records = process_results(
                model_name=cfg.name,
                model_type=cfg.model_type,
                model_id=cfg.model_id,
                n_candidates=n_candidates,
                generation_results=gen_results,
                prompt_source=prompt_source,
                retriever=retriever,
                ground_truth_batch=batch["ground_truth"],
                prompt_indices=prompt_indices,
                k_values=k_values,
            )
            all_records.extend(records)

        df_partial = pd.DataFrame.from_records(all_records)
        df_partial.to_parquet(checkpoint_path, index=False)
        log(
            f"  [{cfg.name}] {prompt_source} | Done. Saved checkpoint ({len(all_records)} records)"
        )

    if cfg.model_type in ("local_hf", "local_hf_ft") and model is not None:
        unload_model(model, tokenizer)

log(f"\nDone. Total records: {len(all_records)}")

## Save final Parquet

In [ ]:
df = pd.DataFrame.from_records(all_records)
prefix = "evaluation_results_rq3" if RUN_MODE == "rq3" else "evaluation_results"
output_path = OUTPUT_PATH / f"{prefix}_{now}.parquet"
df.to_parquet(output_path, index=False)
print(f"Final results saved to {output_path}")
print(f"Shape: {df.shape}")
print(f"Models: {df['model'].unique().tolist()}")
print(f"Candidate counts: {df['n_candidates'].unique().tolist()}")
df.head()

## Summary statistics (mean +/- 95% CI)

In [ ]:
metric_names = ["hit_rate", "mrr", "precision", "recall", "f1", "ndcg"]
rng = np.random.default_rng(SEED)

summary_rows = []
for model_name in df["model"].unique():
    for cc in df[df["model"] == model_name]["n_candidates"].unique():
        mask = (df["model"] == model_name) & (df["n_candidates"] == cc)
        sub = df[mask]
        row = {"model": model_name, "n_candidates": cc, "n": len(sub)}
        for metric in metric_names:
            for k in k_values:
                col = f"{metric}@{k}"
                vals = sub[col].dropna().to_numpy()
                mean, ci_low, ci_high = bootstrap_mean_ci(vals, rng)
                row[f"{col}_mean"] = mean
                row[f"{col}_ci_low"] = ci_low
                row[f"{col}_ci_high"] = ci_high
        summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows)

# Print compact table
for metric in metric_names:
    print(f"\n{metric.upper()}:")
    for k in k_values:
        col = f"{metric}@{k}"
        print(f"  @{k}:")
        for _, row in df_summary.iterrows():
            m = row[f"{col}_mean"]
            lo = row[f"{col}_ci_low"]
            hi = row[f"{col}_ci_high"]
            print(
                f"    {row['model']:20s} | {row['n_candidates']:5s} | "
                f"{m:.4f}  [{lo:.4f}, {hi:.4f}]"
            )

## Visualizations

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")

In [ ]:
def plot_metric_grid(
    data: pd.DataFrame,
    model_order: list[str],
    title: str,
    palette: str = "muted",
    x: str = "n_candidates",
    k: int = 10,
) -> None:
    """Plot 2x3 grid of metric barplots."""
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharey=False)
    for i, metric in enumerate(metric_names):
        ax = axes.flat[i]
        col = f"{metric}@{k}"
        sns.barplot(
            data=data,
            x=x,
            y=col,
            hue="model",
            hue_order=model_order,
            palette=palette,
            ax=ax,
        )
        ax.set_title(f"{metric.upper()}@{k}")
        ax.set_ylabel("")
        if i > 0:
            ax.get_legend().remove()
    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# All models x candidate counts x metrics (FacetGrid barplot)
plot_rows = []
for metric in metric_names:
    for k in k_values:
        col = f"{metric}@{k}"
        for _, row in df.iterrows():
            plot_rows.append(
                {
                    "metric": f"{metric.upper()}@{k}",
                    "model": row["model"],
                    "n_candidates": row["n_candidates"],
                    "value": row[col],
                }
            )

df_plot = pd.DataFrame(plot_rows)

for metric in metric_names:
    subset = df_plot[df_plot["metric"].str.startswith(metric.upper())]
    g = sns.FacetGrid(
        subset,
        col="metric",
        sharey=False,
        height=4,
        aspect=1.3,
    )
    g.map_dataframe(
        sns.barplot,
        x="n_candidates",
        y="value",
        hue="model",
        palette="muted",
    )
    g.add_legend()
    g.figure.suptitle(f"{metric.upper()} across models and candidate counts", y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Llama scaling: 3B vs 8B vs 70B
llama_models = ["Llama-3.2-3B", "Llama-3.1-8B", "Llama-3.3-70B"]
df_llama = df[df["model"].isin(llama_models)].copy()

if not df_llama.empty:
    plot_metric_grid(
        df_llama,
        llama_models,
        "Llama Scaling: 3B vs 8B vs 70B",
        palette="viridis",
    )

In [ ]:
# Sub-10B comparison: Llama-8B vs Qwen-7B vs Gemma-9B
sub10b_models = ["Llama-3.1-8B", "Qwen2.5-7B", "Gemma-2-9B"]
df_sub10b = df[df["model"].isin(sub10b_models)].copy()

if not df_sub10b.empty:
    plot_metric_grid(
        df_sub10b,
        sub10b_models,
        "Sub-10B Comparison: Llama-8B vs Qwen-7B vs Gemma-9B",
        palette="Set2",
    )

In [ ]:
# Qwen base vs FT (c250 only)
qwen_models = ["Qwen2.5-7B", "Qwen2.5-7B-FT"]
df_qwen = df[(df["model"].isin(qwen_models)) & (df["n_candidates"] == "c250")].copy()

if not df_qwen.empty:
    plot_metric_grid(
        df_qwen,
        qwen_models,
        "Qwen2.5-7B: Base vs Fine-Tuned (c250)",
        palette="coolwarm",
        x="model",
    )

    # Print delta
    for metric in metric_names:
        for k in k_values:
            col = f"{metric}@{k}"
            base_val = df_qwen[df_qwen["model"] == "Qwen2.5-7B"][col].mean()
            ft_val = df_qwen[df_qwen["model"] == "Qwen2.5-7B-FT"][col].mean()
            delta = ft_val - base_val
            print(f"{col}: base={base_val:.4f}, FT={ft_val:.4f}, delta={delta:+.4f}")

In [ ]:
# API models comparison
api_models = [
    "infobip-gpt-4-1",
    "infobip-gpt-4-1-mini",
    "gpt-5.2",
    "claude-sonnet-4-6",
    "claude-opus-4-6",
]
df_api = df[df["model"].isin(api_models)].copy()

if not df_api.empty:
    plot_metric_grid(
        df_api,
        api_models,
        "API Models: GPT-4.1, GPT-5.2, Claude Sonnet/Opus 4.6",
        palette="tab10",
    )